# ogbn-proteins: GNN Training with PyTorch Geometric (PyG)

This notebook trains a GNN on the **ogbn-proteins** benchmark from the Open Graph Benchmark (OGB)
using **PyTorch Geometric (PyG)**.

The model supports the following MPNN types:
- `gat`  – Graph Attention Network (no edge features)
- `gate` – GAT with edge features
- `sage` – GraphSAGE
- `gcn`  – Graph Convolutional Network

Reference: *Classic GNNs are Strong Baselines: Reassessing GNNs for Node Classification* (NeurIPS 2024)

## 1. Install Dependencies

Run the cell below **once** if the required packages are not yet installed.

In [ ]:
# Uncomment and run if packages are missing
# !pip install torch
# !pip install torch_geometric
# !pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__)')+cu121.html
# !pip install ogb

## 2. Imports

In [ ]:
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from ogb.nodeproppred import Evaluator, PygNodePropPredDataset
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATConv, GCNConv, SAGEConv
from torch_scatter import scatter

print('All imports successful.')

## 3. Configuration

Edit the variables below to configure the experiment (replaces command-line arguments).

In [ ]:
# ── Device ──────────────────────────────────────────────────────────────────
USE_CPU  = False   # Set True to force CPU mode
GPU_ID   = 0       # GPU device ID (ignored when USE_CPU=True)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED     = 0
N_RUNS   = 1       # Number of independent runs

# ── Model ────────────────────────────────────────────────────────────────────
MPNN       = 'gat' # 'gat' | 'gate' | 'sage' | 'gcn'
N_LAYERS   = 6
N_HEADS    = 6
N_HIDDEN   = 80
USE_LABELS = False # Concatenate training labels as input features
NO_ATTN_DST= False # Disable destination attention (accepted for API consistency; PyG GATConv handles this internally)
JK         = False # Enable Jumping Knowledge (JK) aggregation

# ── Regularisation ───────────────────────────────────────────────────────────
DROPOUT    = 0.25
INPUT_DROP = 0.1
ATTN_DROP  = 0.0
EDGE_DROP  = 0.1

# ── Optimiser ────────────────────────────────────────────────────────────────
LR           = 0.01
WEIGHT_DECAY = 0.0

# ── Training schedule ────────────────────────────────────────────────────────
N_EPOCHS   = 1000
EVAL_EVERY = 5
LOG_EVERY  = 5

# ── Misc ─────────────────────────────────────────────────────────────────────
SAVE_PRED  = False  # Save final predictions to ./output/

# ── Dataset constants (do not change) ────────────────────────────────────────
DATASET_NAME = 'ogbn-proteins'
N_NODE_FEATS = 0    # will be set after preprocessing
N_EDGE_FEATS = 8
N_CLASSES    = 112

# ── Device setup ─────────────────────────────────────────────────────────────
if USE_CPU or not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device(f'cuda:{GPU_ID}')

print(f'Using device: {device}')

## 4. Model Definition

GNN_PyG supports GAT (with/without edge features), GraphSAGE, and GCN.

In [ ]:
class GNN_PyG(nn.Module):
    """Multi-layer GNN (PyG backend) supporting gat / gate / sage / gcn."""

    def __init__(
        self,
        node_feats,
        edge_feats,
        n_classes,
        n_layers,
        n_heads,
        n_hidden,
        edge_emb,
        activation,
        dropout,
        input_drop,
        attn_drop,
        edge_drop,
        use_attn_dst=True,
        allow_zero_in_degree=False,
        mpnn='gat',
        jk=False,
    ):
        super().__init__()
        self.n_layers  = n_layers
        self.n_heads   = n_heads
        self.n_hidden  = n_hidden
        self.n_classes = n_classes
        self.mpnn      = mpnn
        self.jk        = jk
        self.edge_drop = edge_drop

        self.node_encoder = nn.Linear(node_feats, n_hidden)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        # Edge encoder only for 'gate' mode
        self.edge_encoder = nn.ModuleList() if (edge_emb > 0 and mpnn == 'gate') else None

        for i in range(n_layers):
            in_hidden  = n_heads * n_hidden if i > 0 else n_hidden
            out_hidden = n_heads * n_hidden

            if self.edge_encoder is not None:
                self.edge_encoder.append(nn.Linear(edge_feats, edge_emb))

            if mpnn == 'gat':
                self.convs.append(
                    GATConv(
                        in_hidden, n_hidden,
                        heads=n_heads,
                        dropout=attn_drop,
                        concat=True,
                        add_self_loops=False,
                    )
                )
            elif mpnn == 'gate':
                self.convs.append(
                    GATConv(
                        in_hidden, n_hidden,
                        heads=n_heads,
                        dropout=attn_drop,
                        edge_dim=edge_emb,
                        concat=True,
                        add_self_loops=False,
                    )
                )
            elif mpnn == 'sage':
                self.convs.append(SAGEConv(in_hidden, out_hidden))
            else:  # gcn
                self.convs.append(GCNConv(in_hidden, out_hidden, add_self_loops=False))

            self.norms.append(nn.BatchNorm1d(out_hidden))

        self.pred_linear = nn.Linear(n_heads * n_hidden, n_classes)
        self.input_drop  = nn.Dropout(input_drop)
        self.dropout     = nn.Dropout(dropout)
        self.activation  = activation

    def forward(self, x, edge_index, edge_attr=None):
        # Random edge drop during training
        if self.training and self.edge_drop > 0 and edge_index.shape[1] > 0:
            mask       = torch.rand(edge_index.shape[1], device=edge_index.device) >= self.edge_drop
            edge_index = edge_index[:, mask]
            if edge_attr is not None:
                edge_attr = edge_attr[mask]

        h = self.node_encoder(x)
        h = F.relu(h)
        h = self.input_drop(h)

        h_local = []
        h_last  = None

        for i in range(self.n_layers):
            efeat_emb = None
            if self.mpnn == 'gate' and self.edge_encoder is not None:
                efeat_emb = F.relu(self.edge_encoder[i](edge_attr))

            if self.mpnn in ('gat', 'gate'):
                h = self.convs[i](h, edge_index, efeat_emb)
            else:
                h = self.convs[i](h, edge_index)

            if h_last is not None:
                h = h + h_last[: h.shape[0], :]
            h_last = h

            h = self.norms[i](h)
            h = self.activation(h)
            h = self.dropout(h)
            h_local.append(h)

        if self.jk:
            h_local = [t[: h.shape[0], :] for t in h_local]
            h = torch.sum(torch.stack(h_local), dim=0)

        return self.pred_linear(h)


print('Model class defined.')

## 5. Utility Functions

In [ ]:
def set_seed(seed_val=0):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_data(dataset_name):
    dataset_obj = PygNodePropPredDataset(name=dataset_name, root='data/ogb')
    evaluator   = Evaluator(name=dataset_name)
    split_idx   = dataset_obj.get_idx_split()
    train_idx   = split_idx['train']
    val_idx     = split_idx['valid']
    test_idx    = split_idx['test']
    data        = dataset_obj[0]
    return data, train_idx, val_idx, test_idx, evaluator


def preprocess(data, train_idx):
    global N_NODE_FEATS
    # Aggregate edge features to node features via sum of incoming edges
    x = scatter(data.edge_attr, data.edge_index[1], dim=0,
                dim_size=data.num_nodes, reduce='sum')
    data.x = x
    N_NODE_FEATS = data.x.shape[-1]

    # Training labels as additional input features (others stay zero)
    data.train_labels_onehot = torch.zeros(data.num_nodes, N_CLASSES)
    data.train_labels_onehot[train_idx, data.y[train_idx, 0].long()] = 1
    return data


def gen_model():
    n_feats = (N_NODE_FEATS + N_CLASSES) if USE_LABELS else N_NODE_FEATS
    return GNN_PyG(
        n_feats,
        N_EDGE_FEATS,
        N_CLASSES,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        n_hidden=N_HIDDEN,
        edge_emb=16,
        activation=F.relu,
        dropout=DROPOUT,
        input_drop=INPUT_DROP,
        attn_drop=ATTN_DROP,
        edge_drop=EDGE_DROP,
        use_attn_dst=not NO_ATTN_DST,
        mpnn=MPNN,
        jk=JK,
    )


def add_labels(x, train_labels_onehot, idx):
    """Concatenate one-hot training labels to node features for the given indices."""
    labels_onehot = torch.zeros([x.shape[0], N_CLASSES], device=x.device)
    labels_onehot[idx] = train_labels_onehot[idx].to(x.device)
    return torch.cat([x, labels_onehot], dim=-1)


print('Utility functions defined.')

## 6. Training and Evaluation

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    loss_sum, total = 0, 0

    for batch in dataloader:
        batch      = batch.to(device)
        batch_size = batch.batch_size

        if USE_LABELS:
            non_seed_idx = torch.arange(batch_size, batch.x.shape[0], device=device)
            x = add_labels(batch.x, batch.train_labels_onehot, non_seed_idx)
        else:
            x = batch.x

        pred = model(x, batch.edge_index, batch.edge_attr)
        loss = criterion(pred[:batch_size], batch.y[:batch_size].float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * batch_size
        total    += batch_size

    return loss_sum / total


@torch.no_grad()
def evaluate(model, dataloader, labels, train_idx, val_idx, test_idx, criterion, evaluator):
    model.eval()
    preds      = torch.zeros(labels.shape, device=device)
    eval_times = 1

    for _ in range(eval_times):
        for batch in dataloader:
            batch      = batch.to(device)
            batch_size = batch.batch_size

            if USE_LABELS:
                all_idx = torch.arange(batch.x.shape[0], device=device)
                x = add_labels(batch.x, batch.train_labels_onehot, all_idx)
            else:
                x = batch.x

            pred = model(x, batch.edge_index, batch.edge_attr)
            preds[batch.n_id[:batch_size]] += pred[:batch_size]

    preds /= eval_times

    train_loss = criterion(preds[train_idx], labels[train_idx].float()).item()
    val_loss   = criterion(preds[val_idx],   labels[val_idx].float()).item()
    test_loss  = criterion(preds[test_idx],  labels[test_idx].float()).item()

    return (
        evaluator(preds[train_idx], labels[train_idx]),
        evaluator(preds[val_idx],   labels[val_idx]),
        evaluator(preds[test_idx],  labels[test_idx]),
        train_loss, val_loss, test_loss,
        preds,
    )


print('Training/evaluation functions defined.')

## 7. Main Run Function

In [ ]:
def run(data, labels, train_idx, val_idx, test_idx, evaluator, n_running):
    evaluator_wrapper = lambda pred, lbls: evaluator.eval(
        {'y_pred': pred, 'y_true': lbls}
    )['rocauc']

    train_batch_size = (len(train_idx) + 9) // 10

    train_loader = NeighborLoader(
        data,
        num_neighbors=[32] * N_LAYERS,
        batch_size=train_batch_size,
        input_nodes=train_idx.cpu(),
        shuffle=True,
        num_workers=4,
    )

    eval_loader = NeighborLoader(
        data,
        num_neighbors=[100] * N_LAYERS,
        batch_size=65536,
        input_nodes=torch.cat([train_idx.cpu(), val_idx.cpu(), test_idx.cpu()]),
        shuffle=False,
        num_workers=4,
    )

    criterion    = nn.BCEWithLogitsLoss()
    model        = gen_model().to(device)
    optimizer    = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.75, patience=50, verbose=True
    )

    total_time = 0
    best_val_score, final_test_score = 0, 0
    val_score  = 0
    final_pred = None

    for epoch in range(1, N_EPOCHS + 1):
        tic  = time.time()
        loss = train_epoch(model, train_loader, criterion, optimizer)
        toc  = time.time()
        total_time += toc - tic

        if epoch == N_EPOCHS or epoch % EVAL_EVERY == 0 or epoch % LOG_EVERY == 0:
            train_score, val_score, test_score, train_loss, val_loss, test_loss, pred = evaluate(
                model, eval_loader, labels, train_idx, val_idx, test_idx, criterion, evaluator_wrapper
            )

            if val_score > best_val_score:
                best_val_score   = val_score
                final_test_score = test_score
                final_pred       = pred

            if epoch % LOG_EVERY == 0:
                print(
                    f'Epoch: {epoch:04d} | '
                    f'Loss: {loss:.4f} | '
                    f'Train: {100 * train_score:.2f}% | '
                    f'Valid: {100 * val_score:.2f}% | '
                    f'Test: {100 * test_score:.2f}% | '
                    f'Best Valid: {100 * best_val_score:.2f}% | '
                    f'Best Test: {100 * final_test_score:.2f}%'
                )

        lr_scheduler.step(val_score)

    if SAVE_PRED and final_pred is not None:
        os.makedirs('./output', exist_ok=True)
        torch.save(F.softmax(final_pred, dim=1), f'./output/{n_running}.pt')

    return best_val_score, final_test_score


print('Run function defined.')

## 8. Load and Preprocess Data

In [ ]:
print('Loading data ...')
data, train_idx, val_idx, test_idx, evaluator = load_data(DATASET_NAME)

print('Preprocessing ...')
data = preprocess(data, train_idx)

labels = data.y
labels, train_idx, val_idx, test_idx = (
    labels.to(device),
    train_idx.to(device),
    val_idx.to(device),
    test_idx.to(device),
)

print(f'Node features:  {data.x.shape}')
print(f'Labels:         {labels.shape}')
print(f'Train nodes:    {len(train_idx)}')
print(f'Val nodes:      {len(val_idx)}')
print(f'Test nodes:     {len(test_idx)}')
print(f'N_NODE_FEATS (after preprocess): {N_NODE_FEATS}')

## 9. Train the Model

In [ ]:
all_val_scores  = []
all_test_scores = []

for i in range(N_RUNS):
    print(f'\n=== Run {i + 1} / {N_RUNS} ===')
    set_seed(SEED + i)
    val_score, test_score = run(
        data, labels, train_idx, val_idx, test_idx, evaluator, n_running=i + 1
    )
    all_val_scores.append(val_score)
    all_test_scores.append(test_score)
    print(f'Run {i + 1} finished – Val ROC-AUC: {val_score:.4f} | Test ROC-AUC: {test_score:.4f}')

print('\n=== Summary ===')
print(f'Val  ROC-AUC: {np.mean(all_val_scores):.4f} ± {np.std(all_val_scores):.4f}')
print(f'Test ROC-AUC: {np.mean(all_test_scores):.4f} ± {np.std(all_test_scores):.4f}')